# logit-thermo 探索：結晶が始まる歩（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 5〜10 分。
- 探索（予想なし）。A2-2・A2-5・A2-6 で保存した生成のトークン（25 本）から、1 本ずつ「結晶が始まった歩」を拾い、
  同じ記事の 2 サンプルでそろうか（滴定寄りか核寄りか）、始まる手前に助走があるか、を数えて
  `A2_explore/onset_summary.json` と `A2_explore/onset_detail.npz` に書きます。**結果は表示しません。**
- 計算のコード：`scripts/a2_explore_onset.py`（`0b39f997b72f33d3…`）を埋め込み。生成もモデルの読み込みもしません。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo 探索 結晶の始まり（始まる歩・滴定か核か、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 結晶の始まり（始まる歩・滴定か核か、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore onset', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import hashlib, json, math
SRCON = '"""探索：結晶が始まる歩（2026-10-29 18:3x）。確かめには使わない（予想なし）。CPU。生成もモデルの読み込みもしない。\n\n発想（オーナー 2026-10-29 18:27・18:28）：（オーナーの承認・発言。原文は非公開の記録にある）\n問い：\n  (1) 結晶の始まりは「文脈で決まる」（滴定寄り：同じ記事なら 2 サンプルとも同じころに始まる）か、「運」（核寄り：同じ記事でもばらばら）か。\n  (2) 始まる手前に、写しがなだらかに増える助走があるか。\n  (3) Pythia の大きさで、始まり方が変わるか。\n\n材料：A2-2・A2-5・A2-6 で保存した生成のトークン（toks、温度 15 × サンプル 2 × 記事 40 × 512 歩）。探索 T-scan と同じ 25 本。\n写しの印 c_t：続きの t 歩目で終わる 4 トークンの組が、続きの中でそれより前に出ていたら 1（書き出しは数えない。トークナイザを使わないため）。\n始まりの歩：直近 W ＝ 32 歩の c の平均が初めて 0.5 以上になった歩（窓の最後の歩）。512 歩で始まらなければ「始まらない」。\n"""\nimport json\nimport numpy as np\n\nON_RUNS = {\n    \'A2_2\': [\'pythia-1b\', \'qwen2.5-0.5b\', \'qwen2.5-0.5b-inst\', \'qwen2.5-1.5b\', \'olmo2-1b\', \'smollm2-1.7b\', \'smollm2-1.7b-inst\'],\n    \'A2_5\': [\'pythia-70m\', \'pythia-160m\', \'pythia-410m\', \'pythia-1b\', \'pythia-1.4b\', \'pythia-2.8b\',\n             \'pythia-1b@step8000\', \'pythia-1b@step24000\', \'pythia-1b@step64000\'],\n    \'A2_6\': [\'qwen2.5-0.5b\', \'qwen2.5-1.5b\', \'qwen2.5-3b\', \'smollm2-135m\', \'smollm2-360m\', \'smollm2-1.7b\',\n             \'qwen3-0.6b\', \'qwen3-1.7b\', \'qwen3-4b\'],\n}\nON_W, ON_TH = 32, 0.5\nON_PRE, ON_POST, ON_BIN = 160, 32, 8          # 始まりにそろえた平均の幅（歩）と、まとめる幅\nON_HBINS = 4                                  # 始まりやすさ（ハザード）を 128 歩ずつ 4 つで\nON_MIN_BOTH = 8                               # 同じ記事の 2 サンプルの比べ：両方始まった記事がこれ以上のときだけ\n\n\ndef copy_marks(seq):\n    """続きのトークン列 → c（長さ L、0/1）。t 歩目で終わる 4 つ組が、それより前（続きの中）に出ていたら 1。"""\n    s = np.asarray(seq, dtype=np.int64)\n    L = len(s)\n    c = np.zeros(L, np.int8)\n    if L < 4:\n        return c\n    g = np.stack([s[:-3], s[1:-2], s[2:-1], s[3:]], axis=1)          # 4 つ組（語彙が 15 万あるので 1 つの整数にはまとめない）\n    _, first, inv = np.unique(g, axis=0, return_index=True, return_inverse=True)\n    inv = np.asarray(inv).reshape(-1)\n    c[3:] = (first[inv] < np.arange(len(g))).astype(np.int8)\n    return c\n\n\ndef onset_of(c):\n    """→ 始まりの歩（0 始まり、窓の最後の歩）。なければ −1。"""\n    if len(c) < ON_W:\n        return -1\n    r = np.convolve(c.astype(np.float64), np.ones(ON_W), \'valid\') / ON_W   # r[k] は歩 k..k+W−1 の平均\n    k = np.flatnonzero(r >= ON_TH - 1e-12)\n    return int(k[0] + ON_W - 1) if len(k) else -1\n\n\ndef run_stats(toks):\n    """toks：(nT, S, nD, L) → 温度ごとのまとめと、始まりにそろえた平均。"""\n    nT, S, nD, L = toks.shape\n    on = np.full((nT, S, nD), -1, np.int32)\n    lock = np.full((nT, S, nD), np.nan)\n    C = np.zeros((nT, S, nD, L), np.int8)\n    for k in range(nT):\n        for s in range(S):\n            for j in range(nD):\n                c = copy_marks(toks[k, s, j]); C[k, s, j] = c\n                o = onset_of(c); on[k, s, j] = o\n                if o >= 0 and o < L - 1:\n                    r = np.convolve(c.astype(np.float64), np.ones(ON_W), \'valid\') / ON_W\n                    lock[k, s, j] = float(np.mean(r[o - ON_W + 1:] >= ON_TH - 1e-12))\n    per_T = []\n    nb = (ON_PRE + ON_POST) // ON_BIN\n    aligned_sum, aligned_n = np.zeros(nb), 0\n    for k in range(nT):\n        o = on[k]                                        # (S, nD)\n        hit = o >= 0\n        p = float(hit.mean())\n        oo = o[hit]\n        row = dict(p=round(p, 4), n_on=int(hit.sum()))\n        if len(oo):\n            row.update(med=float(np.median(oo)), q25=float(np.percentile(oo, 25)), q75=float(np.percentile(oo, 75)),\n                       cv=round(float(np.std(oo) / np.mean(oo)), 3) if np.mean(oo) > 0 else None,\n                       lock=round(float(np.nanmean(lock[k][hit])), 3))\n        # 片方だけ始まった記事の割合（記事が関係ないなら 2p(1−p)）\n        if S == 2:\n            split = float(np.mean(hit[0] != hit[1]))\n            row.update(split_obs=round(split, 4), split_exp=round(2 * p * (1 - p), 4))\n            both = hit[0] & hit[1]\n            nboth = int(both.sum()); row[\'n_both\'] = nboth\n            if nboth >= ON_MIN_BOTH:\n                a, b = o[0][both].astype(float), o[1][both].astype(float)\n                within = float(np.mean(np.abs(a - b)))\n                diff = np.abs(a[:, None] - b[None, :])\n                between = float(diff[~np.eye(nboth, dtype=bool)].mean())\n                row.update(within=round(within, 1), between=round(between, 1), R=round(within / between, 3) if between > 0 else None)\n        # 始まりやすさ：128 歩ずつ、その区間の頭でまだ始まっていない本のうち、区間の中で始まった割合\n        if p >= 0.1:\n            hz, edges = [], np.linspace(0, L, ON_HBINS + 1).astype(int)\n            flat = o.ravel()\n            for b0, b1 in zip(edges[:-1], edges[1:]):\n                atrisk = (flat < 0) | (flat >= b0)\n                ev = (flat >= b0) & (flat < b1)\n                hz.append(round(float(ev.sum() / atrisk.sum()), 4) if atrisk.sum() else None)\n            row[\'hazard\'] = hz\n        # 始まらなかった本の写しの割合（基準）\n        if (~hit).any():\n            row[\'base_c\'] = round(float(C[k][~hit].mean()), 4)\n        # 始まりにそろえた c の平均（手前に ON_PRE 歩、後ろに ON_POST 歩とれる本だけ）\n        for s in range(S):\n            for j in range(nD):\n                oj = o[s, j]\n                if oj >= ON_PRE and oj + ON_POST + 1 <= L:          # 切り出しの終わり oj + ON_POST + 1 が L を超えないこと\n                    seg = C[k, s, j, oj - ON_PRE + 1: oj + ON_POST + 1].astype(float)\n                    aligned_sum += seg.reshape(nb, ON_BIN).mean(1); aligned_n += 1\n        per_T.append(row)\n    return per_T, (aligned_sum / aligned_n).round(4).tolist() if aligned_n else None, aligned_n, on\n\n\ndef run_onset(base):\n    res = {\'W\': ON_W, \'TH\': ON_TH, \'aligned_bins\': [-ON_PRE + ON_BIN * i for i in range(1, (ON_PRE + ON_POST) // ON_BIN + 1)], \'runs\': {}}\n    allon = {}\n    for folder, names in ON_RUNS.items():\n        for name in names:\n            Z = np.load(f\'{base}/{folder}/{name}.npz\')\n            toks = Z[\'toks\']\n            if \'temps\' in res:\n                assert [float(t) for t in Z[\'temps\']] == res[\'temps\']\n            else:\n                res[\'temps\'] = [float(t) for t in Z[\'temps\']]\n            per_T, al, aln, on = run_stats(toks)\n            res[\'runs\'][f\'{folder}/{name}\'] = dict(per_T=per_T, aligned=al, aligned_n=aln)\n            allon[f\'{folder}/{name}\'] = on\n    return res, allon\n\n\ndef self_test_onset():\n    c = copy_marks([1, 2, 3, 4, 1, 2, 3, 4, 5])\n    assert c.tolist() == [0, 0, 0, 0, 0, 0, 0, 1, 0], c          # 2 回目の 1 2 3 4 の 4 で 1\n    # 40 歩はばらばら、そのあと 7 の周期の単結晶 → 始まりは周期が始まって少しあと\n    rng = np.random.default_rng(0)\n    seq = list(rng.integers(100, 30000, 200)) + [11, 12, 13, 14, 15, 16, 17] * 40\n    c = copy_marks(seq)\n    o = onset_of(c)\n    assert 200 + 3 <= o <= 200 + 7 + 3 + ON_W, o\n    assert onset_of(copy_marks(list(rng.integers(100, 30000, 512)))) == -1\n    # 記事で決まる場合：同じ記事の 2 サンプルが同じ歩で始まる → R ＝ 0、片方だけは 0\n    L, nD = 512, 20\n    toks = np.zeros((1, 2, nD, L), np.int64)\n    for j in range(nD):\n        st = 100 + 15 * j\n        for s in range(2):\n            toks[0, s, j] = np.concatenate([rng.integers(100, 30000, st), np.tile([5, 6, 7, 8, 9], (L - st) // 5 + 1)[:L - st]])\n    per_T, al, aln, on = run_stats(toks)\n    r = per_T[0]\n    assert r[\'p\'] == 1.0 and r[\'split_obs\'] == 0 and r[\'R\'] < 0.05, r\n    assert al is not None and al[-1] > 0.9 and al[0] < 0.1, al\n    # 境目：始まりの歩がちょうど L − ON_POST − 1 と L − ON_POST になる本（2026-10-29 18:35 のエラーの再現）\n    for target in (L - ON_POST - 1, L - ON_POST):\n        for st in range(target - ON_W - 10, target + 1):\n            sq = np.concatenate([rng.integers(100, 30000, st), np.tile([5, 6, 7, 8, 9], L)[:L - st]])\n            if onset_of(copy_marks(sq)) == target:\n                break\n        assert onset_of(copy_marks(sq)) == target, target\n        run_stats(np.tile(sq, (1, 2, 1, 1)))\n    return \'OK\'\n'
assert hashlib.sha256(SRCON.encode()).hexdigest() == '0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9'
exec(SRCON)
def _clean(o):
    if isinstance(o, dict): return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, float) and not math.isfinite(o): return None
    return o
print('自己テスト：', self_test_onset())


In [ ]:
# セル3：数えて保存（結果は表示しない）
import numpy as np
try:
    r, allon = run_onset('/content/drive/MyDrive/logit-thermo')
    r['a2_explore_onset_sha256'] = '0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9'
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo' + '/A2_explore/onset_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    np.savez_compressed('/content/drive/MyDrive/logit-thermo' + '/A2_explore/onset_detail.npz', **{k.replace('/', '__'): v for k, v in allon.items()})
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 結晶の始まりで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：結晶の始まる歩を A2_explore/onset_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
